In [46]:
import os
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1. 경로 설정 및 공통 매핑 데이터
# =========================================================
CSV_PATH = r"C:\Users\user\SLM\02_cuda_aligned\all_logs\evaluation_results_full.csv"
JSON_DIR = r"C:\Users\user\SLM\03_Evaluation_Datasets\results_cognitive"

models = ['Llama_3.2_1B', 'Qwen2.5_1.5B', 'TinyLlama_1.1B']

# [추가됨] 이전 코드와 동일한 물리적 비트 비율 기반 커스텀 X 좌표 
plot_x_map = {
    '16-bit (Base)': 4.0, 
    '8-bit': 3.0, 
    '4-bit': 2.0, 
    '3-bit': 1.5, 
    '2-bit': 1.0
}

# =========================================================
# 2. 데이터 로드 및 병합 
# =========================================================
df_ppl = pd.read_csv(CSV_PATH)
df_ppl['Perplexity'] = pd.to_numeric(df_ppl['Perplexity'], errors='coerce')
csv_bit_mapping = {
    '16-bit': '16-bit (Base)', 'GPTQ_8bit': '8-bit', 
    'GPTQ_4bit': '4-bit', 'GPTQ_3bit': '3-bit', 'GPTQ_2bit': '2-bit'
}
df_ppl['Bit_Unified'] = df_ppl['Bit_Level'].map(csv_bit_mapping)
df_ppl = df_ppl[['Model_Family', 'Bit_Unified', 'Perplexity']].copy()

json_records = []
json_bit_mapping = {
    'Base_16bit': '16-bit (Base)', 'GPTQ_8bit': '8-bit', 
    'GPTQ_4bit': '4-bit', 'GPTQ_3bit': '3-bit', 'GPTQ_2bit': '2-bit'
}

for filename in os.listdir(JSON_DIR):
    if not filename.endswith(".json"): continue
    
    filepath = os.path.join(JSON_DIR, filename)
    name_core = filename.replace("_results.json", "")
    parts = name_core.split('_', 1)
    
    model_name = parts[0].replace('-', '_')
    bit_raw = parts[1]
    bit_unified = json_bit_mapping.get(bit_raw, bit_raw)
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = json.load(f)
        h_acc = content.get('hellaswag', {}).get('acc_norm,none', content.get('hellaswag', {}).get('acc_norm', None))
        p_acc = content.get('piqa', {}).get('acc_norm,none', content.get('piqa', {}).get('acc_norm', None))
        
        json_records.append({
            'Model_Family': model_name,
            'Bit_Unified': bit_unified,
            'HellaSwag_Acc': h_acc * 100 if h_acc else None,
            'PIQA_Acc': p_acc * 100 if p_acc else None
        })

df_merged = pd.merge(df_ppl, pd.DataFrame(json_records), on=['Model_Family', 'Bit_Unified'], how='outer')

# [수정됨] 범주형 정렬 대신 내부 렌더링용 X 좌표 매핑 후 정렬
df_merged['Plot_X'] = df_merged['Bit_Unified'].map(plot_x_map)
df_merged = df_merged.sort_values(by=['Model_Family', 'Plot_X'], ascending=[True, False])

# =========================================================
# 3. Plotly 이중 축(Dual-axis) Subplots 생성
# =========================================================
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=(
        "<b>Llama-3.2-1B</b><br><span style='font-size:13px; color:gray;'>구조적 붕괴 실증</span>", 
        "<b>Qwen2.5-1.5B</b><br><span style='font-size:13px; color:gray;'>RLHF 방어력 실증</span>", 
        "<b>TinyLlama-1.1B</b><br><span style='font-size:13px; color:gray;'>기초 아키텍처 한계</span>"
    ),
    horizontal_spacing=0.12, 
    specs=[[{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}]]
)


metric_styles = {
    'PPL': dict(color='#5DADE2', width=3, dash='dot'),
    'HellaSwag': dict(color='#2E86C1', width=3, dash='solid'),
    'PIQA': dict(color='#1B4F72', width=3, dash='dash')
}

for i, model_name in enumerate(models):
    col = i + 1
    model_data = df_merged[df_merged['Model_Family'] == model_name]
    if model_data.empty: continue
    
    show_leg = (col == 1)
    
    # [수정됨] X축을 Plot_X로 변경하고 Hovertemplate 적용
    fig.add_trace(go.Scatter(
        x=model_data['Plot_X'], y=model_data['Perplexity'],
        mode='lines+markers', name='PPL (낮을수록 우수)',
        line=metric_styles['PPL'], legendgroup='PPL', showlegend=show_leg,
        customdata=model_data['Bit_Unified'],
        hovertemplate="Bit: %{customdata}<br>PPL: %{y:,.2f}<extra></extra>"
    ), row=1, col=col, secondary_y=False)
    
    fig.add_trace(go.Scatter(
        x=model_data['Plot_X'], y=model_data['HellaSwag_Acc'],
        mode='lines+markers', name='HellaSwag (높을수록 우수)',
        line=metric_styles['HellaSwag'], legendgroup='HellaSwag', showlegend=show_leg,
        customdata=model_data['Bit_Unified'],
        hovertemplate="Bit: %{customdata}<br>Acc: %{y:.2f}%<extra></extra>"
    ), row=1, col=col, secondary_y=True)
    
    fig.add_trace(go.Scatter(
        x=model_data['Plot_X'], y=model_data['PIQA_Acc'],
        mode='lines+markers', name='PIQA (높을수록 우수)',
        line=metric_styles['PIQA'], legendgroup='PIQA', showlegend=show_leg,
        customdata=model_data['Bit_Unified'],
        hovertemplate="Bit: %{customdata}<br>Acc: %{y:.2f}%<extra></extra>"
    ), row=1, col=col, secondary_y=True)

# =========================================================
# 4. 마커 스타일링 및 레이아웃 최적화
# =========================================================
for trace in fig.data:
    if isinstance(trace, go.Scatter):
        trace.update(
            marker=dict(symbol='circle', color='white', size=9, line=dict(color=trace.line.color, width=2.5))
        )

fig.update_layout(
    title=dict(
        text='<b>양자화 압력에 따른 언어적 통계(PPL)와 인지적 지능(Reasoning)의 붕괴 괴리</b>',
        font=dict(size=22), x=0.5, xanchor='center', y=0.95
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    hovermode='x unified',
    legend=dict(
        orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5,
        bgcolor='rgba(255, 255, 255, 0.9)', bordercolor='lightgray', borderwidth=1
    ),
    margin=dict(t=130, b=100, l=60, r=60),
    width=1600 # [추가됨] 전체 너비를 1600으로 강제하여 이중 축 간격 확보
)

for col in range(1, 4):
    # [수정됨] X축을 선형 스케일(Linear)로 두고, 커스텀 텍스트 렌더링 적용
    fig.update_xaxes(
        type='linear',
        autorange="reversed",
        tickmode='array',
        tickvals=[4.0, 3.0, 2.0, 1.5, 1.0],  # 커스텀 좌표 위치
        ticktext=['16-bit (Base)', '8-bit', '4-bit', '3-bit', '2-bit'], # 화면 표시 텍스트
        gridcolor='lightgray', zeroline=False, row=1, col=col
    )
    
    fig.update_yaxes(
        title_text="Perplexity (Log Scale)" if col == 1 else "", 
        type='log', tickformat=",.0f", dtick=1, 
        gridcolor='lightgray', secondary_y=False, row=1, col=col
    )
    
    fig.update_yaxes(
        title_text="Accuracy (%)" if col == 3 else "", 
        range=[20, 80], 
        showgrid=False, secondary_y=True, row=1, col=col
    )
    
    # [수정됨] 4-bit 강조선의 위치를 커스텀 좌표값(2.0)으로 변경
    fig.add_vline(x=2.0, line_width=2, line_dash='dash', line_color='gray', opacity=0.4, row=1, col=col)

fig.show()

In [72]:
import os
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# =========================================================
# 1. 경로 설정 및 데이터 준비
# =========================================================
CSV_PATH = r"C:\Users\user\SLM\02_cuda_aligned\all_logs\evaluation_results_full.csv"
JSON_DIR = r"C:\Users\user\SLM\03_Evaluation_Datasets\results_cognitive"

models = ['Llama_3.2_1B', 'Qwen2.5_1.5B', 'TinyLlama_1.1B']
color_map = {
    'Llama_3.2_1B': '#5A9BD5',    
    'TinyLlama_1.1B': '#E67E22',   
    'Qwen2.5_1.5B': '#9B59B6'
}

# =========================================================
# 2. 데이터 추출 및 전처리
# =========================================================
df_ppl = pd.read_csv(CSV_PATH)
df_ppl['Perplexity'] = pd.to_numeric(df_ppl['Perplexity'], errors='coerce')
df_ppl['Perplexity'] = df_ppl['Perplexity'].replace([np.inf, -np.inf], np.nan).clip(upper=100000) 

bit_map_csv = {'16-bit': '16-bit', 'GPTQ_8bit': '8-bit', 'GPTQ_4bit': '4-bit', 'GPTQ_3bit': '3-bit', 'GPTQ_2bit': '2-bit'}
df_ppl['Bit_Level'] = df_ppl['Bit_Level'].map(bit_map_csv)

json_records = []
bit_map_json = {'Base_16bit': '16-bit', 'GPTQ_8bit': '8-bit', 'GPTQ_4bit': '4-bit', 'GPTQ_3bit': '3-bit', 'GPTQ_2bit': '2-bit'}

for filename in os.listdir(JSON_DIR):
    if not filename.endswith(".json"): continue
    parts = filename.replace("_results.json", "").split('_', 1)
    model_name = parts[0].replace('-', '_')
    bit_level = bit_map_json.get(parts[1], parts[1])
    
    with open(os.path.join(JSON_DIR, filename), 'r', encoding='utf-8') as f:
        content = json.load(f)
        h_acc = content.get('hellaswag', {}).get('acc_norm,none', content.get('hellaswag', {}).get('acc_norm', None))
        if h_acc:
            json_records.append({'Model_Family': model_name, 'Bit_Level': bit_level, 'HellaSwag_Acc': h_acc * 100})

df_merged = pd.merge(df_ppl, pd.DataFrame(json_records), on=['Model_Family', 'Bit_Level'], how='inner')

bit_order = ['16-bit', '8-bit', '4-bit', '3-bit', '2-bit']
df_merged['Bit_Level'] = pd.Categorical(df_merged['Bit_Level'], categories=bit_order, ordered=True)
df_merged = df_merged.sort_values(by=['Model_Family', 'Bit_Level'])

# =========================================================
# 3. 수학적 범위 계산 (텍스트 잘림 방지용 동적 스케일링)
# =========================================================
min_ppl = df_merged['Perplexity'].min()
max_ppl = df_merged['Perplexity'].max()

x_range_min = np.log10(min_ppl) - 0.25 
x_range_max = np.log10(max_ppl) + 0.15 

# =========================================================
# 4. 궤적 시각화 (기본 라인 및 마커 생성)
# =========================================================
fig = go.Figure()

for model in models:
    model_data = df_merged[df_merged['Model_Family'] == model].dropna(subset=['Perplexity', 'HellaSwag_Acc'])
    if model_data.empty: continue
    
    fig.add_trace(go.Scatter(
        x=model_data['Perplexity'], 
        y=model_data['HellaSwag_Acc'],
        mode='lines+markers', 
        name=model,
        customdata=model_data['Bit_Level'], 
        hovertemplate="<b>Quantization: %{customdata}</b><br>PPL: %{x:.2f}<br>HellaSwag: %{y:.2f}%<extra></extra>",
        line=dict(color=color_map[model], width=3),
        marker=dict(symbol='circle', color='white', size=10, line=dict(color=color_map[model], width=2.5))
    ))

# =========================================================
# 5. 개별 데이터 포인트 Annotation (픽셀 단위 정밀 위치 제어)
# =========================================================
for model in models:
    model_data = df_merged[df_merged['Model_Family'] == model].dropna(subset=['Perplexity', 'HellaSwag_Acc'])
    
    for _, row in model_data.iterrows():
        bit = row['Bit_Level']
        x_val = row['Perplexity']
        y_val = row['HellaSwag_Acc']
        
        # 텍스트 및 위치 기본값 초기화
        text_label = ""
        x_shift = 0
        y_shift = 15 
        x_anchor = 'center'
        y_anchor = 'bottom'
        
        # [모델별 맞춤형 라벨링 및 좌표 할당 로직]
        if model == 'Qwen2.5_1.5B':
            if bit == '16-bit':
                text_label = '16~8-bit'
            elif bit == '8-bit':
                continue
            elif bit in ['4-bit', '3-bit', '2-bit']:
                text_label = bit.replace('-bit', '-bit')
                x_shift = -15       
                y_shift = 0         
                x_anchor = 'right'
                y_anchor = 'middle'
                
        elif model == 'TinyLlama_1.1B':
            if bit in ['16-bit', '8-bit']:
                continue
            elif bit == '4-bit':
                text_label = '16~4-bit'
                x_shift = 12
                y_shift = 15        
            elif bit in ['3-bit', '2-bit']:
                text_label = bit.replace('-bit', '-bit')
                x_shift = 12
                y_shift = 15
                
        elif model == 'Llama_3.2_1B':
            if bit == '16-bit':
                text_label = '16~8-bit'
                x_shift = 12
            elif bit == '8-bit':
                continue
            elif bit in ['4-bit', '3-bit', '2-bit']:
                text_label = bit.replace('-bit', '-bit')
                y_shift = 15
        
        # 설정된 라벨이 있는 경우에만 그래프에 그리기
        if text_label:
            fig.add_annotation(
                x=np.log10(x_val),  # Log Scale 동기화
                y=y_val,            
                text=f"<b>{text_label}</b>",
                showarrow=False,
                xshift=x_shift,
                yshift=y_shift,
                xanchor=x_anchor,
                yanchor=y_anchor,
                font=dict(size=12, color=color_map[model])
            )

# =========================================================
# 6. 연구적 지표 보완 및 레이아웃 최적화
# =========================================================
fig.add_hline(
    y=25, line_dash="dash", line_color="rgba(128, 128, 128, 0.7)", line_width=1.5,
    annotation_text="Random Chance (25%)", annotation_position="bottom right",
    annotation_font=dict(size=12, color="gray")
)

base_width = 1200
calculated_height = int(base_width * (9 / 16))

fig.update_layout(
    width=base_width,
    height=calculated_height,
    
    title=dict(
        text='<b>통계적 유창성(PPL)과 논리적 추론(Reasoning)의 상관관계 붕괴 궤적</b><br>' + 
             '<span style="font-size: 14px; color: gray;">문맥적 상식 추론 능력 (HellaSwag Accuracy, %) ↓ 하단일수록 악화</span>',
        font=dict(size=22), x=0.5, xanchor='center', y=0.92
    ),
    xaxis=dict(
        title='<b>언어 모델링 손실도 (Perplexity, Log Scale) → 우측일수록 악화</b>',
        type='log',
        range=[x_range_min, x_range_max], 
        tickformat=",.0f", dtick=1, gridcolor='lightgray',
        zeroline=False
    ),
    yaxis=dict(
        title='<b>HellaSwag Accuracy(%)</b>',
        range=[20, 75], 
        dtick=5, # [수정] 10 단위에서 5 단위로 변경 (20, 25, 30, 35 ... 80)
        gridcolor='lightgray', zeroline=False
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        bgcolor='rgba(255, 255, 255, 0.9)'
    ),
    margin=dict(t=120, b=80, l=80, r=80) 
)

fig.show()

In [7]:
import os
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# =========================================================
# 1. 경로 설정 및 데이터 준비
# =========================================================
CSV_PATH = r"C:\Users\user\SLM\02_cuda_aligned\all_logs\evaluation_results_full.csv"
JSON_DIR = r"C:\Users\user\SLM\03_Evaluation_Datasets\results_cognitive"

models = ['Llama_3.2_1B', 'Qwen2.5_1.5B', 'TinyLlama_1.1B']
color_map = {
    'Llama_3.2_1B': '#5A9BD5',    
    'TinyLlama_1.1B': '#E67E22',   
    'Qwen2.5_1.5B': '#9B59B6'
}

# =========================================================
# 2. 데이터 추출 및 전처리
# =========================================================
df_ppl = pd.read_csv(CSV_PATH)
df_ppl['Perplexity'] = pd.to_numeric(df_ppl['Perplexity'], errors='coerce')
df_ppl['Perplexity'] = df_ppl['Perplexity'].replace([np.inf, -np.inf], np.nan).clip(upper=100000) 

bit_map_csv = {'16-bit': '16-bit', 'GPTQ_8bit': '8-bit', 'GPTQ_4bit': '4-bit', 'GPTQ_3bit': '3-bit', 'GPTQ_2bit': '2-bit'}
df_ppl['Bit_Level'] = df_ppl['Bit_Level'].map(bit_map_csv)

json_records = []
bit_map_json = {'Base_16bit': '16-bit', 'GPTQ_8bit': '8-bit', 'GPTQ_4bit': '4-bit', 'GPTQ_3bit': '3-bit', 'GPTQ_2bit': '2-bit'}

for filename in os.listdir(JSON_DIR):
    if not filename.endswith(".json"): continue
    parts = filename.replace("_results.json", "").split('_', 1)
    model_name = parts[0].replace('-', '_')
    bit_level = bit_map_json.get(parts[1], parts[1])
    
    with open(os.path.join(JSON_DIR, filename), 'r', encoding='utf-8') as f:
        content = json.load(f)
        h_acc = content.get('hellaswag', {}).get('acc_norm,none', content.get('hellaswag', {}).get('acc_norm', None))
        if h_acc:
            json_records.append({'Model_Family': model_name, 'Bit_Level': bit_level, 'HellaSwag_Acc': h_acc * 100})

df_merged = pd.merge(df_ppl, pd.DataFrame(json_records), on=['Model_Family', 'Bit_Level'], how='inner')

# [수정됨] 2-bit 데이터 필터링 및 카테고리 순서에서 제외
df_merged = df_merged[df_merged['Bit_Level'] != '2-bit']
bit_order = ['16-bit', '8-bit', '4-bit', '3-bit']
df_merged['Bit_Level'] = pd.Categorical(df_merged['Bit_Level'], categories=bit_order, ordered=True)
df_merged = df_merged.sort_values(by=['Model_Family', 'Bit_Level'])

# =========================================================
# 3. 수학적 범위 계산 (텍스트 잘림 방지용 동적 스케일링)
# =========================================================
min_ppl = df_merged['Perplexity'].min()
max_ppl = df_merged['Perplexity'].max()

# 2-bit가 빠지면서 max_ppl 값이 작아졌을 수 있으므로 여백을 유지
x_range_min = np.log10(min_ppl) - 0.25 
x_range_max = np.log10(max_ppl) + 0.15 

# =========================================================
# 4. 궤적 시각화 (기본 라인 및 마커 생성)
# =========================================================
fig = go.Figure()

for model in models:
    model_data = df_merged[df_merged['Model_Family'] == model].dropna(subset=['Perplexity', 'HellaSwag_Acc'])
    if model_data.empty: continue
    
    fig.add_trace(go.Scatter(
        x=model_data['Perplexity'], 
        y=model_data['HellaSwag_Acc'],
        mode='lines+markers', 
        name=model,
        customdata=model_data['Bit_Level'], 
        hovertemplate="<b>Quantization: %{customdata}</b><br>PPL: %{x:.2f}<br>HellaSwag: %{y:.2f}%<extra></extra>",
        line=dict(color=color_map[model], width=3),
        marker=dict(symbol='circle', color='white', size=10, line=dict(color=color_map[model], width=2.5))
    ))

# =========================================================
# 5. 개별 데이터 포인트 Annotation (픽셀 단위 정밀 위치 제어)
# =========================================================
for model in models:
    model_data = df_merged[df_merged['Model_Family'] == model].dropna(subset=['Perplexity', 'HellaSwag_Acc'])
    
    for _, row in model_data.iterrows():
        bit = row['Bit_Level']
        x_val = row['Perplexity']
        y_val = row['HellaSwag_Acc']
        
        # 텍스트 및 위치 기본값 초기화
        text_label = ""
        x_shift = 0
        y_shift = 15 
        x_anchor = 'center'
        y_anchor = 'bottom'
        
        # [수정됨] 2-bit 조건 제거 및 모델별 맞춤형 라벨링 제어
        if model == 'Qwen2.5_1.5B':
            if bit == '16-bit':
                text_label = '16~8-bit'
            elif bit == '8-bit':
                continue
            elif bit in ['4-bit', '3-bit']:
                text_label = bit
                x_shift = -15       
                y_shift = 0         
                x_anchor = 'right'
                y_anchor = 'middle'
                
        elif model == 'TinyLlama_1.1B':
            if bit in ['16-bit', '8-bit']:
                continue
            elif bit == '4-bit':
                text_label = '16~4-bit'
                x_shift = 15
                y_shift = 17        
            elif bit == '3-bit':
                text_label = bit
                x_shift = 12
                y_shift = 15
                
        elif model == 'Llama_3.2_1B':
            if bit == '16-bit':
                text_label = '16~8-bit'
                x_shift = 12
            elif bit == '8-bit':
                continue
            elif bit in ['4-bit', '3-bit']:
                text_label = bit
                y_shift = 15
        
        # 설정된 라벨이 있는 경우에만 그래프에 그리기
        if text_label:
            fig.add_annotation(
                x=np.log10(x_val),  # Log Scale 동기화
                y=y_val,            
                text=f"<b>{text_label}</b>",
                showarrow=False,
                xshift=x_shift,
                yshift=y_shift,
                xanchor=x_anchor,
                yanchor=y_anchor,
                font=dict(size=12, color=color_map[model])
            )

# =========================================================
# 6. 연구적 지표 보완 및 레이아웃 최적화
# =========================================================
fig.add_hline(
    y=25, line_dash="dash", line_color="rgba(128, 128, 128, 0.7)", line_width=1.5,
    annotation_text="Random Chance (25%)", annotation_position="bottom right",
    annotation_font=dict(size=12, color="gray")
)

base_width = 1200
calculated_height = int(base_width * (9 / 16))

fig.update_layout(
    width=base_width,
    height=calculated_height,
    
    title=dict(
        text='<b>통계적 유창성(PPL)과 논리적 추론(Reasoning)의 상관관계 붕괴 궤적</b><br>' + 
             '<span style="font-size: 14px; color: gray;">문맥적 상식 추론 능력 (HellaSwag Accuracy, %) ↓ 하단일수록 악화</span>',
        font=dict(size=22), x=0.5, xanchor='center', y=0.92
    ),
    xaxis=dict(
        title='<b>언어 모델링 손실도 (Perplexity, Log Scale) → 우측일수록 악화</b>',
        type='log',
        range=[x_range_min, x_range_max], 
        tickformat=",.0f", dtick=1, gridcolor='lightgray',
        zeroline=False
    ),
    yaxis=dict(
        title='<b>HellaSwag Accuracy(%)</b>',
        range=[52, 70], 
        dtick=5, 
        gridcolor='lightgray', zeroline=False
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        bgcolor='rgba(255, 255, 255, 0.9)'
    ),
    margin=dict(t=120, b=80, l=80, r=80) 
)

fig.show()

In [71]:
import os
import json
import math
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1. 경로 설정 및 공통 매핑 데이터
# =========================================================
CSV_PATH = r"C:\Users\user\SLM\02_cuda_aligned\all_logs\evaluation_results_full.csv"
JSON_DIR = r"C:\Users\user\SLM\03_Evaluation_Datasets\results_cognitive"

color_map = {
    'Llama_3.2_1B': '#5A9BD5',    
    'TinyLlama_1.1B': '#E67E22',   
    'Qwen2.5_1.5B': '#9B59B6'
}

bit_numeric_map = {
    '16-bit': 16.0, 'Base_16bit': 16.0,
    'GPTQ_8bit': 8.0, 
    'GPTQ_4bit': 4.0, 
    'GPTQ_3bit': 3.0, 
    'GPTQ_2bit': 2.0
}

# [핵심] 시각적 간격을 강제로 1:1로 맞추기 위한 X축 렌더링용 커스텀 좌표
# 16->8, 8->4는 간격 1.0 / 4->3, 3->2는 간격 0.5로 완벽히 동일하게 설정
plot_x_map = {
    16.0: 4.0, 
    8.0: 3.0, 
    4.0: 2.0, 
    3.0: 1.5, 
    2.0: 1.0
}

# =========================================================
# 2. 데이터 로드 및 병합 
# =========================================================
df_ppl = pd.read_csv(CSV_PATH)
df_ppl['Perplexity'] = pd.to_numeric(df_ppl['Perplexity'], errors='coerce')
df_ppl['Bit_Num'] = df_ppl['Bit_Level'].map(bit_numeric_map)
df_ppl = df_ppl[['Model_Family', 'Bit_Num', 'Perplexity']]

json_records = []
for filename in os.listdir(JSON_DIR):
    if not filename.endswith(".json"): continue
    
    parts = filename.replace("_results.json", "").split('_', 1)
    model_name = parts[0].replace('-', '_')
    bit_raw = parts[1]
    bit_num = bit_numeric_map.get(bit_raw, None)
    
    with open(os.path.join(JSON_DIR, filename), 'r', encoding='utf-8') as f:
        content = json.load(f)
        h_acc = content.get('hellaswag', {}).get('acc_norm,none', content.get('hellaswag', {}).get('acc_norm', None))
        p_acc = content.get('piqa', {}).get('acc_norm,none', content.get('piqa', {}).get('acc_norm', None))
        
        json_records.append({
            'Model_Family': model_name, 'Bit_Num': bit_num,
            'HellaSwag_Acc': h_acc * 100 if h_acc else None,
            'PIQA_Acc': p_acc * 100 if p_acc else None
        })

df_merged = pd.merge(df_ppl, pd.DataFrame(json_records), on=['Model_Family', 'Bit_Num'], how='outer')
df_merged = df_merged.sort_values(by=['Model_Family', 'Bit_Num'], ascending=[True, False])

# 렌더링용 X좌표 컬럼 추가
df_merged['Plot_X'] = df_merged['Bit_Num'].map(plot_x_map)

# =========================================================
# 3. Plotly 시각화 (커스텀 X축 비율 적용)
# =========================================================
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=(
        "<b>[언어 모델링 성능]</b> Perplexity<br><span style='font-size:13px; color:gray;'>(Log Scale, 낮을수록 우수)</span>", 
        "<b>[문맥 상식 추론]</b> HellaSwag<br><span style='font-size:13px; color:gray;'>(acc_norm, 높을수록 우수)</span>", 
        "<b>[물리 상식 추론]</b> PIQA<br><span style='font-size:13px; color:gray;'>(acc_norm, 높을수록 우수)</span>"
    ),
    horizontal_spacing=0.08
)

models = df_merged['Model_Family'].dropna().unique()

for model in models:
    model_data = df_merged[df_merged['Model_Family'] == model]
    if model_data.empty or model not in color_map: continue
    
    line_style = dict(color=color_map[model], width=3)
    
    # x값을 'Plot_X' (커스텀 좌표)로 변경
    fig.add_trace(go.Scatter(
        x=model_data['Plot_X'], y=model_data['Perplexity'],
        mode='lines+markers', name=model,
        line=line_style, legendgroup=model,
        customdata=model_data['Bit_Num'],
        hovertemplate="Bit: %{customdata}-bit<br>PPL: %{y:.2f}<extra></extra>"
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=model_data['Plot_X'], y=model_data['HellaSwag_Acc'],
        mode='lines+markers', name=model,
        line=line_style, legendgroup=model, showlegend=False,
        customdata=model_data['Bit_Num'],
        hovertemplate="Bit: %{customdata}-bit<br>Acc: %{y:.2f}%<extra></extra>"
    ), row=1, col=2)
    
    fig.add_trace(go.Scatter(
        x=model_data['Plot_X'], y=model_data['PIQA_Acc'],
        mode='lines+markers', name=model,
        line=line_style, legendgroup=model, showlegend=False,
        customdata=model_data['Bit_Num'],
        hovertemplate="Bit: %{customdata}-bit<br>Acc: %{y:.2f}%<extra></extra>"
    ), row=1, col=3)

# =========================================================
# 4. 레이아웃 최적화
# =========================================================
for trace in fig.data:
    if isinstance(trace, go.Scatter):
        trace.update(marker=dict(symbol='circle', color='white', size=10, line=dict(color=trace.line.color, width=2.5)))

fig.update_layout(
    title=dict(
        text='<b>실제 물리적 압축 비율(Bit-width) 간격에 따른 인지적 붕괴 절벽(Cliff) 실증</b>',
        font=dict(size=22), x=0.5, xanchor='center', y=0.92
    ),
    plot_bgcolor='white', paper_bgcolor='white', hovermode='x unified',
    legend=dict(
        title='<b></b>', orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5,
        bgcolor='rgba(255, 255, 255, 0.9)', bordercolor='lightgray', borderwidth=1
    ),
    margin=dict(t=120, b=100, l=60, r=40),
    width=1600 
)

fig.update_yaxes(type='log', tickformat=",.0f", dtick=1, gridcolor='lightgray', row=1, col=1)
fig.update_yaxes(title_text="Accuracy (%)", gridcolor='lightgray', row=1, col=2)
fig.update_yaxes(title_text="Accuracy (%)", gridcolor='lightgray', row=1, col=3)

# [수정됨] 커스텀 Linear 스케일을 적용하여 시각적 간격을 정확히 통제
for col in range(1, 4):
    fig.update_xaxes(
        type='linear', 
        autorange="reversed", 
        tickmode='array',
        tickvals=[4.0, 3.0, 2.0, 1.5, 1.0],  # 커스텀 좌표 위치
        ticktext=['16-bit', '8-bit', '4-bit', '3-bit', '2-bit'], # 화면 표시 텍스트
        gridcolor='lightgray', zeroline=False, row=1, col=col
    )
    # 4-bit 붕괴 임계점 강조선 (커스텀 좌표인 2.0 위치에 생성)
    fig.add_vline(x=2.0, line_width=2, line_dash='dash', line_color='red', opacity=0.5, row=1, col=col)

fig.show()